<a href="https://colab.research.google.com/github/OuyangXuelili/BigData/blob/main/train_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -U transformers accelerate datasets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 22.5 MB/s eta 0:00:00


In [1]:
from google.colab import files
uploaded = files.upload()

Saving sampled_dataset_labeled_audited.csv to sampled_dataset_labeled_audited (1).csv


In [2]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, precision_recall_curve

In [3]:
df = pd.read_csv("sampled_dataset_labeled_audited.csv")

df["text_input"] = df["title"].fillna("") + " " + df["text"].fillna("")

df = df[["text_input", "quality_complaint"]]
df = df.rename(columns={"quality_complaint": "label"})

df.head()

,text_input,label
0,works great product works great,0
1,three stars don't pair with echo voice control...,0
2,works great in the hallway works great in the ...,0
3,three stars they work but the shipping was rea...,0
4,only one card out of a 2 pack is reliable boug...,0


In [4]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df["label"], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42
)

In [5]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df.reset_index(drop=True))
test_ds = Dataset.from_pandas(test_df.reset_index(drop=True))

In [6]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text_input"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

In [7]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1": f1_score(labels, preds)}

In [10]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    logging_steps=50
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [13]:
trainer.train()

Step,Training Loss
50,0.595876
100,0.387531
150,0.387571
200,0.340450
250,0.254400
300,0.213314
350,0.225788
400,0.205875
450,0.164504
500,0.135184


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=657, training_loss=0.2499770092546849, metrics={'train_runtime': 237.0321, 'train_samples_per_second': 44.298, 'train_steps_per_second': 2.772, 'total_flos': 695453842944000.0, 'train_loss': 0.2499770092546849, 'epoch': 3.0})

In [14]:
preds = trainer.predict(test_ds)

y_true = preds.label_ids
logits = preds.predictions
y_pred = np.argmax(logits, axis=1)

print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           0       0.95      0.92      0.94       531
           1       0.82      0.89      0.85       219

    accuracy                           0.91       750
   macro avg       0.89      0.91      0.89       750
weighted avg       0.91      0.91      0.91       750



In [15]:
probs = torch.nn.functional.softmax(
    torch.tensor(logits), dim=1
).numpy()[:,1]

precision, recall, thresholds = precision_recall_curve(y_true, probs)

f1_scores = 2 * precision * recall / (precision + recall + 1e-8)

best_idx = np.argmax(f1_scores)
best_thresh = thresholds[min(best_idx, len(thresholds)-1)]

print("Best threshold:", best_thresh)
print("Best F1:", f1_scores[best_idx])

Best threshold: 0.9354198
Best F1: 0.8738317707036641


In [16]:
final_preds = (probs >= best_thresh).astype(int)

print(classification_report(y_true, final_preds))

              precision    recall  f1-score   support

           0       0.94      0.96      0.95       531
           1       0.89      0.85      0.87       219

    accuracy                           0.93       750
   macro avg       0.92      0.91      0.91       750
weighted avg       0.93      0.93      0.93       750



In [17]:
trainer.save_model("distilbert_model")
tokenizer.save_pretrained("distilbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilbert_model/tokenizer_config.json', 'distilbert_model/tokenizer.json')

In [18]:
!zip -r distilbert_model.zip distilbert_model
from google.colab import files
files.download("distilbert_model.zip")

  adding: distilbert_model/ (stored 0%)
  adding: distilbert_model/model.safetensors (deflated 8%)
  adding: distilbert_model/training_args.bin (deflated 53%)
  adding: distilbert_model/config.json (deflated 49%)
  adding: distilbert_model/tokenizer.json (deflated 71%)
  adding: distilbert_model/tokenizer_config.json (deflated 42%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>